# Tâche 3 - Visualisation & Insights (Notebook)
**Auteur : Kakpo Christamour**

Notebook daté 05/10 reprenant `Tache3_ANIP_WPP.py`. Produit graphiques narratifs, cartes Plotly et CSV prêt pour Power BI.

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
print('Python', sys.version)
print('pandas', pd.__version__)

In [ ]:
# Paths
input_file = 'outputs/tache2/WPP2024_DEMOGRAPHIC_ENRICHED.csv'
output_dir = 'outputs/tache3'
graphs_dir = os.path.join(output_dir, 'graphs')
maps_dir = os.path.join(output_dir, 'maps')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(maps_dir, exist_ok=True)
print('Input:', input_file)
print('Output dir:', output_dir)

In [ ]:
# Charger le dataset
df = pd.read_csv(input_file)
print('Dataset chargé :', df.shape)

In [ ]:
# Nettoyage et KPI
df.fillna({'pop_growth_rate':0,'sex_ratio':1,'regional_dev_index':0}, inplace=True)
df['world_population'] = df.groupby('year')['population_total'].transform('sum')
df['population_pct'] = df['population_total'] / df['world_population'] * 100
def categorize_population(pop):
    if pop < 1_000: return '<1M'
    elif pop < 10_000: return '1M-10M'
    elif pop < 50_000: return '10M-50M'
    elif pop < 100_000: return '50M-100M'
    else: return '>100M'
df['pop_category'] = df['population_total'].apply(categorize_population)
print('KPI applied')

In [ ]:
# Agrégation region x year
agg_cols = ['population_total','pop_growth_rate','sex_ratio','regional_dev_index','population_pct']
df_agg = df.groupby(['region','year'])[agg_cols].mean().reset_index()
powerbi_csv = os.path.join(output_dir, 'WPP2024_POWERBI_READY.csv')
df_agg.to_csv(powerbi_csv, index=False)
print('Power BI CSV saved:', powerbi_csv)

In [ ]:
# Graphiques narratifs
plt.figure(figsize=(10,5))
df.groupby('year')['population_total'].sum().plot()
plt.title('Évolution population mondiale (1950-2023)')
plt.ylabel('Population (milliers)')
plt.savefig(os.path.join(graphs_dir,'population_mondiale.png'))
plt.close()
plt.figure(figsize=(12,6))
sns.lineplot(data=df_agg, x='year', y='regional_dev_index', hue='region', legend=False)
plt.title('Indice de développement régional moyen (1950-2023)')
plt.savefig(os.path.join(graphs_dir,'regional_dev_index.png'))
plt.close()
plt.figure(figsize=(8,6))
sns.heatmap(df_agg[['population_total','pop_growth_rate','sex_ratio','regional_dev_index']].corr(), annot=True, cmap='coolwarm')
plt.title('Corrélations principales')
plt.savefig(os.path.join(graphs_dir,'correlation_heatmap.png'))
plt.close()
print('Graphiques sauvés dans', graphs_dir)

In [ ]:
# Cartes choroplèthes (nécessite iso3)
if 'iso3' not in df.columns:
    raise ValueError(
)
df_2023 = df[df['year'] == 2023]
fig_pop = px.choropleth(df_2023, locations='iso3', color='population_total', hover_name='region', hover_data=['population_total','pop_growth_rate','regional_dev_index'], color_continuous_scale='Viridis', title='Population totale par pays (2023)')
fig_pop.write_html(os.path.join(maps_dir,'population_2023.html'))
fig_pop.write_image(os.path.join(maps_dir,'population_2023.png'))
fig_index = px.choropleth(df_2023, locations='iso3', color='regional_dev_index', hover_name='region', hover_data=['population_total','pop_growth_rate','sex_ratio'], color_continuous_scale='Plasma', title='Indice de développement régional (2023)')
fig_index.write_html(os.path.join(maps_dir,'regional_dev_index_2023.html'))
fig_index.write_image(os.path.join(maps_dir,'regional_dev_index_2023.png'))
fig_anim = px.choropleth(df, locations='iso3', color='population_total', hover_name='region', animation_frame='year', color_continuous_scale='Viridis', title='Évolution population totale par pays (1950-2023)')
fig_anim.write_html(os.path.join(maps_dir,'population_1950_2023.html'))
print('Cartes sauvegardées dans', maps_dir)